In [1]:
from transformers import AutoConfig, AutoModelForImageClassification
import torch

from hf_utils import convert_hf_swinv2_state_dict
from paritycheck import forward_hf_debug, forward_my_debug, compare_debug_dicts
from swin2_utils import SwinV2Cfg, MySwinV2

In [2]:
hf_name = "microsoft/swinv2-tiny-patch4-window16-256"
hf_cfg = AutoConfig.from_pretrained(hf_name)
hf_model = AutoModelForImageClassification.from_pretrained(hf_name)
hf_state = hf_model.state_dict()

my_cfg = SwinV2Cfg(
    image_size=hf_cfg.image_size,
    patch_size=hf_cfg.patch_size,
    num_channels=hf_cfg.num_channels,
    embed_dim=hf_cfg.embed_dim,
    depths=tuple(hf_cfg.depths),
    num_heads=tuple(hf_cfg.num_heads),
    window_size=hf_cfg.window_size,
    pretrained_window_sizes=tuple(hf_cfg.pretrained_window_sizes),
    mlp_ratio=hf_cfg.mlp_ratio,
    qkv_bias=hf_cfg.qkv_bias,
    hidden_dropout_prob=hf_cfg.hidden_dropout_prob,
    attention_probs_dropout_prob=hf_cfg.attention_probs_dropout_prob,
    drop_path_rate=hf_cfg.drop_path_rate,
    layer_norm_eps=hf_cfg.layer_norm_eps,
    num_labels=hf_cfg.num_labels,
)
my_model = MySwinV2(my_cfg, num_classes=my_cfg.num_labels)
my_model = MySwinV2(my_cfg, num_classes=my_cfg.num_labels)

converted_state = convert_hf_swinv2_state_dict(hf_state, my_model)

missing, unexpected = my_model.load_state_dict(converted_state, strict=False)

Loading weights:   0%|          | 0/245 [00:00<?, ?it/s]

In [5]:
my_model.eval()
hf_model.eval()

x = torch.randn(2, 3, 256, 256)

my_dbg = forward_my_debug(my_model, x)
hf_dbg = forward_hf_debug(hf_model, x)

compare_debug_dicts(my_dbg, hf_dbg)

patch_embed                    shape=(2, 4096, 96) max=0.000000 mean=0.000000
pos_drop                       shape=(2, 4096, 96) max=0.000000 mean=0.000000
layer0.block0                  shape=(2, 4096, 96) max=0.000000 mean=0.000000
layer0.block1                  shape=(2, 4096, 96) max=0.000000 mean=0.000000
layer0.out                     shape=(2, 4096, 96) max=0.000000 mean=0.000000
layer0.downsample              shape=(2, 1024, 192) max=0.000000 mean=0.000000
layer1.block0                  shape=(2, 1024, 192) max=0.000000 mean=0.000000
layer1.block1                  shape=(2, 1024, 192) max=0.000000 mean=0.000000
layer1.out                     shape=(2, 1024, 192) max=0.000000 mean=0.000000
layer1.downsample              shape=(2, 256, 384) max=0.000000 mean=0.000000
layer2.block0                  shape=(2, 256, 384) max=0.000000 mean=0.000000
layer2.block1                  shape=(2, 256, 384) max=0.000000 mean=0.000000
layer2.block2                  shape=(2, 256, 384) max=0.000

In [7]:
with torch.no_grad():
    dbg_logits = forward_my_debug(my_model, x)["logits"]
    normal_logits = my_model(x)
    y_hf = hf_model(pixel_values=x).logits
    y_my = my_model(x)

print("max abs diff:", (y_hf - y_my).abs().max().item())
print("mean abs diff:", (y_hf - y_my).abs().mean().item())

print("my debug vs my normal:",
      (dbg_logits - normal_logits).abs().max().item(),
      (dbg_logits - normal_logits).abs().mean().item())

max abs diff: 0.0
mean abs diff: 0.0
my debug vs my normal: 0.0 0.0
